Этот код конвертирует HS-коды в ОКВЭД2 для санкций

In [3]:
import pandas as pd

SANCTIONS_FILE = 'export trade sanctions.xlsx'
# SANCTIONS_FILE = 'import trade sanctions.xlsx'
MAPPING_FILE = 'HS to OKPD2 mapping.xlsx'
OUTPUT_FILE = 'export_sanctions_okpd2.xlsx'
# OUTPUT_FILE = 'import_sanctions_okpd2.xlsx'

sanctions = pd.read_excel(SANCTIONS_FILE)
mapping = pd.read_excel(MAPPING_FILE, converters={'OKPD2': lambda x: str(x).strip()})

sanctions = sanctions.rename(columns={'HS code': 'HS'})
mapping = mapping.rename(columns={'HS code': 'HS'})

date_cols = sanctions.columns.tolist()
date_cols.remove('HS')

sanctions_long = sanctions.melt(
    id_vars='HS',
    value_vars=date_cols,
    var_name='period',
    value_name='sanction_value'
)

merged = sanctions_long.merge(
    mapping,
    on='HS',
    how='left'
)

okpd2_agg = (
    merged
    .groupby(['OKPD2', 'period'], as_index=False)
    ['sanction_value']
    .mean()
)

okpd2_wide = okpd2_agg.pivot(
    index='OKPD2',
    columns='period',
    values='sanction_value'
).reset_index()

okpd2_wide = okpd2_wide[['OKPD2'] + date_cols]

okpd2_wide.to_excel(OUTPUT_FILE, index=False)